# 07 Results Freeze

Final reviewer evidence matrix and artifact freeze.

## Setup

Clone/pull latest code into `/content/ECG-RAMBA`, restore verified Drive mirror artifacts, and define a logging command helper. Notebook 07 is self-contained: you do not need to run Notebook 00 first. If you skip this setup cell and start from **Required Input Check**, that cell contains a direct-run guard that performs the same lightweight setup before freezing final evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path('/content/ECG-RAMBA')

os.chdir('/content')
if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a git checkout. Rename it or use a fresh runtime.')
if not REPO_DIR.exists():
    print(f'$ git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"')
    subprocess.run(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', shell=True, check=True)

os.chdir(REPO_DIR)

def run(cmd, check=True, log_path=None, tail_lines=120, cwd=None):
    run_cwd = Path(cwd) if cwd else Path.cwd()
    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = run_cwd / 'reports' / 'revision' / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f'notebook_command_{time.strftime("%Y%m%d_%H%M%S")}.log'
    else:
        log_path = Path(log_path)
        if not log_path.is_absolute():
            log_path = run_cwd / log_path
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(run_cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        rc = proc.wait()
    print(f'Command log: {log_path}')
    if check and rc:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(rc, cmd)
    return subprocess.CompletedProcess(cmd, rc)

run('git fetch origin', check=False)
run(f'git checkout {BRANCH}', check=True)
run(f'git pull --ff-only --autostash origin {BRANCH}', check=True)

if MIRROR_REVISION_ROOT.exists():
    run(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        log_path='reports/revision/logs/final_evidence_restore.log',
    )
else:
    print('Mirror not found; continuing with repo-local reports/revision:', MIRROR_REVISION_ROOT)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)
run('git rev-parse --short HEAD', check=False)
run('git status --short --branch', check=False)


## Required Input Check

Validate the revision-plan CSV files and the final evidence inputs before building the final matrix. This cell is safe to run directly after opening Notebook 07: if Setup was skipped, it mounts Drive, clones/pulls the active repo, defines `run()`, and restores required evidence artifacts before continuing.


In [ ]:
# Notebook 07 direct-run guard: allows starting from this cell without Notebook 00 or the Setup cell.
from pathlib import Path
import json
import os
import subprocess
import sys
import time

if (
    'run' not in globals()
    or 'DRIVE_ROOT' not in globals()
    or 'MIRROR_REVISION_ROOT' not in globals()
    or not Path.cwd().as_posix().endswith('/content/ECG-RAMBA')
):
    print('Notebook 07 setup context missing; running embedded direct setup guard.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Drive mount skipped or already mounted: {exc}')

    DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
    MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
    REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
    BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
    REPO_DIR = Path('/content/ECG-RAMBA')

    os.chdir('/content')
    if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a git checkout. Rename it or use a fresh runtime.')
    if not REPO_DIR.exists():
        print(f'$ git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"')
        subprocess.run(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', shell=True, check=True)

    os.chdir(REPO_DIR)

    def run(cmd, check=True, log_path=None, tail_lines=120, cwd=None):
        run_cwd = Path(cwd) if cwd else Path.cwd()
        print(f'$ {cmd}', flush=True)
        if log_path is None:
            log_dir = run_cwd / 'reports' / 'revision' / 'logs'
            log_dir.mkdir(parents=True, exist_ok=True)
            log_path = log_dir / f'notebook_command_{time.strftime("%Y%m%d_%H%M%S")}.log'
        else:
            log_path = Path(log_path)
            if not log_path.is_absolute():
                log_path = run_cwd / log_path
            log_path.parent.mkdir(parents=True, exist_ok=True)

        with log_path.open('w', encoding='utf-8') as log:
            proc = subprocess.Popen(
                cmd,
                shell=True,
                cwd=str(run_cwd),
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding='utf-8',
                errors='replace',
                bufsize=1,
            )
            assert proc.stdout is not None
            for line in proc.stdout:
                print(line, end='')
                log.write(line)
                log.flush()
            rc = proc.wait()
        print(f'Command log: {log_path}')
        if check and rc:
            lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
            for line in lines[-tail_lines:]:
                print(line)
            raise subprocess.CalledProcessError(rc, cmd)
        return subprocess.CompletedProcess(cmd, rc)

    run('git fetch origin', check=False)
    run(f'git checkout {BRANCH}', check=True)
    run(f'git pull --ff-only --autostash origin {BRANCH}', check=True)
    print('Direct setup guard complete.')
    print('cwd       :', Path.cwd())
    print('drive_root:', DRIVE_ROOT)
    print('repo_dir  :', REPO_DIR)
    print('branch    :', BRANCH)
else:
    print('Notebook 07 setup context already present; continuing with existing repo/session.')

import pandas as pd

revision_root = Path('reports/revision')
plan_csvs = [
    Path('docs/revision_plan/experiment_registry.csv'),
    Path('docs/revision_plan/claim_evidence_map.csv'),
    Path('docs/revision_plan/task_board.csv'),
]
for csv_path in plan_csvs:
    df = pd.read_csv(csv_path)
    print(f'{csv_path}: rows={len(df)} cols={len(df.columns)}')

required_final_inputs = {
    'calibration': revision_root / 'metrics' / 'calibration_ci_oof_final_ema_predictions.json',
    'pooling': revision_root / 'metrics' / 'pooling_sensitivity.csv',
    'baseline': revision_root / 'metrics' / 'baseline_summary.csv',
    'component': revision_root / 'metrics' / 'component_check_summary.json',
    'hrv_domain': revision_root / 'metrics' / 'hrv_domain_summary.csv',
    'robustness': revision_root / 'metrics' / 'robustness_summary.csv',
    'paired_minirocket': revision_root / 'metrics' / 'paired_full_vs_minirocket_comparison.json',
    'paired_resnet': revision_root / 'metrics' / 'paired_full_vs_resnet_comparison.json',
    'a0_status': revision_root / 'a0_resolution_status.json',
    'claim_map': Path('docs/revision_plan/claim_evidence_map.csv'),
    'task_board': Path('docs/revision_plan/task_board.csv'),
}
optional_final_inputs = {
    'paired_raw_mamba': revision_root / 'metrics' / 'paired_full_vs_raw_mamba_comparison.json',
    'paired_transformer': revision_root / 'metrics' / 'paired_full_vs_transformer_comparison.json',
    'paired_hybrid_morphology': revision_root / 'metrics' / 'paired_full_vs_hybrid_morphology_comparison.json',
    'hybrid_morphology_summary': revision_root / 'metrics' / 'hybrid_morphology_baseline_summary.json',
    'claim_readiness_gates': revision_root / 'metrics' / 'claim_readiness_gates.json',
    'claim_readiness_table': revision_root / 'tables' / 'table_claim_readiness_gates.csv',
    'external_protocol_gate_summary': revision_root / 'metrics' / 'external_protocol_gate_summary.csv',
    'representation_evidence_status': revision_root / 'metrics' / 'representation_evidence_status.json',
    'representation_probe_summary': revision_root / 'metrics' / 'representation_probe_summary.json',
    'representation_probe_table': revision_root / 'tables' / 'table_representation_probe.csv',
    'representation_cka_table': revision_root / 'tables' / 'table_representation_cka.csv',
    'representation_probe_manifest': revision_root / 'manifests' / 'representation_probe_manifest.json',
    'fewshot_ptbxl_summary': revision_root / 'metrics' / 'fewshot_ptbxl_summary.csv',
    'fewshot_ptbxl_table': revision_root / 'tables' / 'table_fewshot_ptbxl.csv',
    'fewshot_ptbxl_bootstrap': revision_root / 'metrics' / 'fewshot_ptbxl_bootstrap.json',
    'fewshot_ptbxl_manifest': revision_root / 'manifests' / 'fewshot_ptbxl_run_manifest.json',
    'fewshot_cpsc2021_summary': revision_root / 'metrics' / 'fewshot_cpsc2021_summary.csv',
    'fewshot_cpsc2021_table': revision_root / 'tables' / 'table_fewshot_cpsc2021.csv',
    'fewshot_cpsc2021_bootstrap': revision_root / 'metrics' / 'fewshot_cpsc2021_bootstrap.json',
    'fewshot_cpsc2021_manifest': revision_root / 'manifests' / 'fewshot_cpsc2021_run_manifest.json',
    'fewshot_georgia_summary': revision_root / 'metrics' / 'fewshot_georgia_summary.csv',
    'fewshot_georgia_table': revision_root / 'tables' / 'table_fewshot_georgia.csv',
    'fewshot_georgia_bootstrap': revision_root / 'metrics' / 'fewshot_georgia_bootstrap.json',
    'fewshot_georgia_manifest': revision_root / 'manifests' / 'fewshot_georgia_run_manifest.json',
    'robustness_multicomparator_summary': revision_root / 'metrics' / 'robustness_multicomparator_summary.csv',
    'robustness_multicomparator_pairwise': revision_root / 'metrics' / 'robustness_multicomparator_pairwise.json',
    'robustness_multicomparator_table': revision_root / 'tables' / 'table_robustness_multicomparator.csv',
    'robustness_multicomparator_manifest': revision_root / 'manifests' / 'robustness_multicomparator_manifest.json',
}
def collect_missing_final_inputs(verbose=True):
    missing = []
    for label, path in required_final_inputs.items():
        exists = path.exists()
        if verbose:
            print(f'{label:18}: exists={exists} size={path.stat().st_size if exists else None} path={path}')
        if not exists:
            missing.append(f'{label}={path}')
    return missing

missing_inputs = collect_missing_final_inputs(verbose=True)
if missing_inputs:
    print('\nRequired inputs are missing from the active repo. Attempting verified Drive mirror restore before failing.')
    if 'MIRROR_REVISION_ROOT' not in globals():
        raise RuntimeError('MIRROR_REVISION_ROOT is undefined. Run the Setup cell first.')
    if not MIRROR_REVISION_ROOT.exists():
        raise FileNotFoundError(f'Mirror root does not exist: {MIRROR_REVISION_ROOT}')
    run(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        log_path='reports/revision/logs/final_evidence_required_restore.log',
    )
    print('\nRechecking required final evidence inputs after mirror restore:')
    missing_inputs = collect_missing_final_inputs(verbose=True)
    if missing_inputs:
        print('\nVerified mirror manifest did not restore every required input. Trying direct path fallback from Drive mirror/repo roots.')
        import shutil

        def _candidate_source_roots():
            roots = []
            if 'MIRROR_REVISION_ROOT' in globals():
                roots.append(('verified_mirror_path', Path(MIRROR_REVISION_ROOT)))
            drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
            roots.append(('drive_revision_artifacts', Path(drive_root) / 'revision_artifacts' / 'reports' / 'revision'))
            return roots

        restored_direct, unresolved_direct = [], []
        for label, destination in required_final_inputs.items():
            destination = Path(destination)
            if destination.exists() and destination.stat().st_size > 0:
                continue
            try:
                rel_to_revision = destination.relative_to(revision_root)
            except ValueError:
                unresolved_direct.append(f'{label}={destination} (not under reports/revision; direct fallback skipped)')
                continue
            copied = False
            for source_label, source_root in _candidate_source_roots():
                source = source_root / rel_to_revision
                if source.exists() and source.stat().st_size > 0:
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(source, destination)
                    restored_direct.append(f'{label}: {source_label}:{source} -> {destination}')
                    copied = True
                    break
            if not copied:
                unresolved_direct.append(f'{label}={destination}')
        print(f'Direct final-input fallback restore: restored={len(restored_direct)} unresolved={len(unresolved_direct)}')
        for item in restored_direct:
            print(' - restored', item)
        for item in unresolved_direct:
            print(' - unresolved', item)
        print('\nRechecking required final evidence inputs after direct fallback restore:')
        missing_inputs = collect_missing_final_inputs(verbose=True)
if missing_inputs:
    raise FileNotFoundError('Missing required final evidence inputs after mirror restore/direct fallback: ' + '; '.join(missing_inputs))

# Cross-notebook provenance guard: Notebook 07 must not freeze final tables
# if calibration or paired comparisons still point to an older Full OOF file.
from scripts.revision.common import sha256_file

current_oof_path = revision_root / 'predictions' / 'oof_final_ema_predictions.npz'
current_freeze_path = revision_root / 'manifests' / 'oof_final_ema_freeze_manifest.json'
current_oof_sha = sha256_file(current_oof_path)
current_freeze_sha = sha256_file(current_freeze_path)
current_freeze = json.loads(current_freeze_path.read_text(encoding='utf-8'))
print('Current frozen OOF SHA     :', current_oof_sha)
print('Current freeze manifest SHA:', current_freeze_sha)

calibration_payload = json.loads(required_final_inputs['calibration'].read_text(encoding='utf-8'))
calibration_errors = []
if calibration_payload.get('predictions_sha256') != current_oof_sha:
    calibration_errors.append(
        'calibration predictions_sha256 mismatch: '
        f"json={calibration_payload.get('predictions_sha256')} current={current_oof_sha}"
    )
if calibration_payload.get('freeze_manifest_sha256') != current_freeze_sha:
    calibration_errors.append(
        'calibration freeze_manifest_sha256 mismatch: '
        f"json={calibration_payload.get('freeze_manifest_sha256')} current={current_freeze_sha}"
    )
expected_shape = {
    'y_prob': [current_freeze.get('validated_records'), current_freeze.get('n_classes')],
    'y_true': [current_freeze.get('validated_records'), current_freeze.get('n_classes')],
}
if calibration_payload.get('shape') != expected_shape:
    calibration_errors.append(f"calibration shape mismatch: json={calibration_payload.get('shape')} expected={expected_shape}")
if calibration_errors:
    print('\n'.join(' - ' + item for item in calibration_errors))
    raise RuntimeError(
        'Calibration artifact is stale for the current frozen OOF. '
        'Rerun Notebook 03, publish the mirror, then rerun Notebook 04 and Notebook 07.'
    )
print('Calibration provenance guard: OK')

paired_required = {
    'MiniRocket-only': required_final_inputs['paired_minirocket'],
    'ResNet1D/CNN': required_final_inputs['paired_resnet'],
}
paired_optional = {
    'Raw Mamba': optional_final_inputs['paired_raw_mamba'],
    'Transformer ECG': optional_final_inputs['paired_transformer'],
}

def _paired_protocol_compatible(label, payload):
    inputs = payload.get('inputs', {})
    freeze_input = inputs.get('freeze_manifest') or {}
    full_input = inputs.get('full_predictions') or {}
    full_metadata = full_input.get('metadata') or {}
    expected_records = int(current_freeze.get('validated_records', -1))
    expected_classes = int(current_freeze.get('n_classes', -1))
    expected_fingerprint = current_freeze.get('dataset_record_order_fingerprint')
    freeze_contract_ok = (
        freeze_input.get('checkpoint_kind') == 'final_ema'
        and int(freeze_input.get('validated_records', -1)) == expected_records
        and int(freeze_input.get('n_classes', -1)) == expected_classes
        and freeze_input.get('dataset_record_order_fingerprint') == expected_fingerprint
    )
    full_contract_ok = (
        full_metadata.get('checkpoint_kind') == 'final_ema'
        and full_metadata.get('dataset_record_order_fingerprint') == expected_fingerprint
        and full_metadata.get('protocol') == 'fold_final_ema_power_mean_v2_q3_threshold_0.5'
        and float(full_metadata.get('threshold', 0.5)) == 0.5
    )
    current_full_metrics = {}
    current_full_metrics.update(calibration_payload.get('metrics') or {})
    current_full_metrics.update(calibration_payload.get('calibration') or {})
    metric_mismatches = []
    for metric_name, row in (payload.get('metrics') or {}).items():
        if metric_name not in current_full_metrics:
            continue
        observed = row.get('full_value')
        expected = current_full_metrics.get(metric_name)
        if observed is None or expected is None:
            metric_mismatches.append(f'{metric_name}: missing observed={observed} expected={expected}')
            continue
        if abs(float(observed) - float(expected)) > 1e-12:
            metric_mismatches.append(f'{metric_name}: paired={observed} current={expected}')
    return freeze_contract_ok and full_contract_ok and not metric_mismatches, {
        'freeze_contract_ok': freeze_contract_ok,
        'full_contract_ok': full_contract_ok,
        'metric_mismatches': metric_mismatches,
    }


def _check_paired_payload(label, path, required=True):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        if required:
            raise FileNotFoundError(f'Missing required paired comparison for {label}: {path}')
        print(f'Paired {label}: absent/deferred')
        return
    payload = json.loads(path.read_text(encoding='utf-8'))
    inputs = payload.get('inputs', {})
    full_sha = (inputs.get('full_predictions') or {}).get('sha256')
    freeze_sha = (inputs.get('freeze_manifest') or {}).get('sha256')
    if full_sha == current_oof_sha and freeze_sha == current_freeze_sha:
        print(f'Paired {label} provenance guard: OK')
        return
    compatible, details = _paired_protocol_compatible(label, payload)
    if compatible:
        print(
            f'Paired {label} provenance guard: stable-protocol reuse OK. '
            f'Container SHA changed full_sha={full_sha} current={current_oof_sha}; '
            f'freeze_sha={freeze_sha} current={current_freeze_sha}, but final_ema record/class/fingerprint contract '
            'and full metric values match current calibration.'
        )
        return
    raise RuntimeError(
        f'Paired {label} comparison is stale for current OOF. '
        f'full_sha={full_sha} current={current_oof_sha}; '
        f'freeze_sha={freeze_sha} current={current_freeze_sha}; details={details}. '
        'Rerun Notebook 04 paired comparison cells, then rerun Notebook 07.'
    )

for label, path in paired_required.items():
    _check_paired_payload(label, path, required=True)
for label, path in paired_optional.items():
    _check_paired_payload(label, path, required=False)

def restore_optional_final_inputs_from_drive():
    import shutil

    def _candidate_source_roots():
        roots = []
        if 'MIRROR_REVISION_ROOT' in globals():
            roots.append(('verified_mirror_path', Path(MIRROR_REVISION_ROOT)))
        drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
        roots.append(('drive_revision_artifacts', Path(drive_root) / 'revision_artifacts' / 'reports' / 'revision'))
        return roots

    restored, still_missing = [], []
    for label, destination in optional_final_inputs.items():
        destination = Path(destination)
        if destination.exists() and destination.stat().st_size > 0:
            continue
        try:
            rel_to_revision = destination.relative_to(revision_root)
        except ValueError:
            still_missing.append(f'{label}={destination} (not under reports/revision)')
            continue
        copied = False
        for source_label, source_root in _candidate_source_roots():
            source = source_root / rel_to_revision
            if source.exists() and source.stat().st_size > 0:
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, destination)
                restored.append(f'{label}: {source_label}:{source} -> {destination}')
                copied = True
                break
        if not copied:
            drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
            flat_source = Path(drive_root) / 'final_evidence_tables' / destination.name
            if flat_source.exists() and flat_source.stat().st_size > 0:
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(flat_source, destination)
                restored.append(f'{label}: final_evidence_tables:{flat_source} -> {destination}')
                copied = True
        if not copied:
            still_missing.append(f'{label}={destination}')
    print(f'Optional final-input path fallback restore: restored={len(restored)} still_missing={len(still_missing)}')
    for item in restored:
        print(' - restored', item)
    if still_missing:
        print('Optional inputs still absent after fallback; these remain deferred unless their notebook cells are run:')
        for item in still_missing:
            print(' - missing', item)


restore_optional_final_inputs_from_drive()

print('Optional final evidence inputs:')
for label, path in optional_final_inputs.items():
    exists = path.exists()
    print(f'{label:18}: exists={exists} size={path.stat().st_size if exists else None} path={path}')

# Guard against stale Colab code: if few-shot artifacts exist, the final generator
# must know how to ingest them into C06 and claim_guidance.
generator_path = Path('scripts/revision/13_final_evidence_matrix.py')
generator_source = generator_path.read_text(encoding='utf-8')
required_generator_tokens = [
    'def summarize_fewshot',
    'def combine_fewshot_summaries',
    'fewshot_dataset_summaries',
    'fewshot_summary = combine_fewshot_summaries',
    '"fewshot_summary": fewshot_summary',
    '"fewshot": (',
    '"paired_transformer":',
    'transformer_paired_key_numbers',
    '"paired_hybrid_morphology":',
    'hybrid_paired_key_numbers',
    '"hybrid_morphology": (',
    '"claim_readiness_gates": (',
    '"claim_readiness_gates": claim_readiness_gates',
]
missing_generator_tokens = [token for token in required_generator_tokens if token not in generator_source]
if missing_generator_tokens:
    raise RuntimeError(
        'Final evidence generator is stale and cannot ingest the current optional evidence artifacts. '
        'Pull latest main / rerun Notebook 00 setup, then rerun Notebook 07. Missing tokens: '
        + ', '.join(missing_generator_tokens)
    )
print('Final evidence generator optional-evidence support: OK')

fewshot_datasets_for_final = ['ptbxl', 'cpsc2021', 'georgia']
fewshot_complete_datasets = []
fewshot_artifacts_present = False
for dataset in fewshot_datasets_for_final:
    dataset_paths = [
        optional_final_inputs[f'fewshot_{dataset}_summary'],
        optional_final_inputs[f'fewshot_{dataset}_table'],
        optional_final_inputs[f'fewshot_{dataset}_bootstrap'],
        optional_final_inputs[f'fewshot_{dataset}_manifest'],
    ]
    dataset_artifacts_present = all(path.exists() and path.stat().st_size > 0 for path in dataset_paths)
    print(f'Few-shot artifact set complete for {dataset}:', dataset_artifacts_present)
    if dataset_artifacts_present:
        fewshot_manifest = json.loads(optional_final_inputs[f'fewshot_{dataset}_manifest'].read_text(encoding='utf-8'))
        print(f'Few-shot manifest status for {dataset}:', fewshot_manifest.get('status'))
        if fewshot_manifest.get('status') != 'complete':
            raise RuntimeError(f'Few-shot manifest exists for {dataset} but is not complete; rerun Notebook 02 few-shot cell or remove stale artifact.')
        fewshot_complete_datasets.append(dataset)
fewshot_artifacts_present = bool(fewshot_complete_datasets)
print('Few-shot complete datasets:', fewshot_complete_datasets)

paired_resnet = json.loads(required_final_inputs['paired_resnet'].read_text(encoding='utf-8'))
resnet_metrics = paired_resnet.get('metrics', {})
print('Paired ResNet interpretations:')
for metric in ['pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro']:
    row = resnet_metrics.get(metric, {})
    print(f"  {metric}: {row.get('interpretation')} full={row.get('full_value')} comparator={row.get('comparator_value')}")

paired_raw_path = optional_final_inputs['paired_raw_mamba']
if paired_raw_path.exists():
    paired_raw = json.loads(paired_raw_path.read_text(encoding='utf-8'))
    raw_metrics = paired_raw.get('metrics', {})
    print('Paired Raw Mamba interpretations:')
    for metric in ['pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro']:
        row = raw_metrics.get(metric, {})
        print(f"  {metric}: {row.get('interpretation')} full={row.get('full_value')} comparator={row.get('comparator_value')}")
else:
    print('Paired Raw Mamba optional input is absent; final matrix will keep Raw Mamba incomplete until Notebook 04 produces it.')

expected_external_datasets = ['ptbxl', 'georgia', 'cpsc2021']
external_gate_path = optional_final_inputs['external_protocol_gate_summary']
if external_gate_path.exists():
    external_gate = pd.read_csv(external_gate_path)
    external_gate_columns = [
        'dataset',
        'status',
        'protocol_gate_passed',
        'manuscript_ready',
        'issue_count',
        'reused_existing',
        'gate_cache_key',
        'prediction_sha256',
        'slice_prediction_sha256',
        'gate_json',
        'gate_manifest',
    ]
    display_columns = [col for col in external_gate_columns if col in external_gate.columns]
    print('External protocol gate summary:')
    display(external_gate[display_columns])
    gate_pass_mask = external_gate.get('protocol_gate_passed', pd.Series(False, index=external_gate.index)).astype(str).str.lower().isin({'true', '1', 'yes'})
    present_datasets = sorted(external_gate['dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    passed_datasets = sorted(external_gate.loc[gate_pass_mask, 'dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    blocked_datasets = sorted(external_gate.loc[~gate_pass_mask, 'dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    deferred_datasets = [dataset for dataset in expected_external_datasets if dataset not in present_datasets]
    print('External gate passed datasets:', ', '.join(passed_datasets) if passed_datasets else 'none')
    print('External gate blocked datasets in summary:', ', '.join(blocked_datasets) if blocked_datasets else 'none')
    print('External deferred datasets absent from gate summary:', ', '.join(deferred_datasets) if deferred_datasets else 'none')
    print('External gate supporting artifacts:')
    for dataset in sorted(external_gate['dataset'].astype(str).tolist()) if 'dataset' in external_gate.columns else []:
        supporting = {
            'gate_json': revision_root / 'metrics' / f'external_{dataset}_protocol_gate.json',
            'label_mapping': revision_root / 'tables' / f'table_external_{dataset}_label_mapping.csv',
            'metrics_table': revision_root / 'tables' / f'table_external_{dataset}_metrics.csv',
            'gate_manifest': revision_root / 'manifests' / f'external_{dataset}_protocol_gate_manifest.json',
        }
        for label, path in supporting.items():
            print(f'  {dataset:8} {label:14}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
else:
    print('External protocol gate summary absent; external datasets remain experimental/deferred in final evidence.')


## Claim Readiness Gates

Generate a lightweight blocker ledger for optional or unsupported claims before building the final matrix. This keeps full-HRV, clinical-readiness, broad-superiority, transformer, hybrid, and learned-comparator robustness wording auditable.


In [ ]:
# No embedded stale fallback: the repository script itself is the auditable source.
claim_readiness_script = Path('scripts/revision/28_claim_readiness_gates.py')
required_tokens = [
    'group_safe_score_calibration_v2',
    'true_fewshot_frozen_encoder_head_adaptation',
    'external_learned_comparator_audit',
    'representation_probe_fold_safe_v3_projection_and_fold_audit',
    'marked_highlighted_manuscript',
]
source_text = claim_readiness_script.read_text(encoding='utf-8') if claim_readiness_script.exists() else ''
missing_tokens = [token for token in required_tokens if token not in source_text]
if missing_tokens:
    raise RuntimeError(
        'Claim-readiness gate is stale. Pull/push the current main branch or copy the updated script into the active repo. '
        f'Missing tokens: {missing_tokens}'
    )
run('python -u scripts/revision/28_claim_readiness_gates.py', log_path='reports/revision/logs/claim_readiness_gates.log')
for item in [
    revision_root / 'metrics' / 'claim_readiness_gates.json',
    revision_root / 'tables' / 'table_claim_readiness_gates.csv',
    revision_root / 'manifests' / 'claim_readiness_gates_manifest.json',
]:
    print(f'{item}: exists={item.exists()} size={item.stat().st_size if item.exists() else None}')


## Final Evidence Matrix

Build claim-level evidence, blocker status, robustness claim rows, and reviewer-safe wording from frozen artifacts.

In [ ]:
import pandas as pd

run('python -u scripts/revision/13_final_evidence_matrix.py --strict', log_path='reports/revision/logs/final_evidence_matrix.log')
matrix = pd.read_csv('reports/revision/tables/table_final_evidence_matrix.csv')
safe = pd.read_csv('reports/revision/tables/table_final_safe_wording.csv')
robust = pd.read_csv('reports/revision/tables/table_final_robustness_claims.csv')
blockers = pd.read_csv('reports/revision/tables/table_final_blocker_status.csv')
payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
guidance = payload.get('claim_guidance', {})
for token in ['claim_readiness_gates', 'hybrid_morphology', 'fewshot', 'external_comparators', 'marked_manuscript']:
    if token not in guidance:
        raise RuntimeError(f'Final generator is stale; missing claim guidance token: {token}')
legacy = payload.get('legacy_row_split_score_calibration', {})
if any(item.get('claim_ready') is True for item in legacy.values() if isinstance(item, dict)):
    raise RuntimeError('Legacy row-split score-calibration artifact was incorrectly marked claim-ready.')
print('Final evidence matrix validated. final_ready_for_rebuttal=', payload.get('final_ready_for_rebuttal'))
print('Group-safe score calibration:', payload.get('group_safe_score_calibration_summary', {}))
print('True frozen-encoder head adaptation:', payload.get('true_fewshot_head_adaptation_summary', {}))
print('External learned-comparator audit:', payload.get('external_learned_comparator_audit', {}))
display(matrix[['claim_id', 'claim_topic', 'evidence_status', 'key_numbers', 'blocker']])
display(safe[['claim_id', 'evidence_status', 'safe_wording']])
display(robust[['stress_test', 'metric', 'degradation_interpretation', 'stressed_performance_interpretation']])
display(blockers[['blocker_id', 'resolution_status', 'restriction']])


## Validation Gate

Fail if the final evidence matrix is incomplete or if blocked/limited claims are accidentally marked as fully supported.

In [ ]:
payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
manifest = json.loads(Path('reports/revision/manifests/final_evidence_matrix_manifest.json').read_text(encoding='utf-8'))
required = [
    Path('reports/revision/metrics/final_evidence_matrix.json'),
    Path('reports/revision/tables/table_final_evidence_matrix.csv'),
    Path('reports/revision/tables/table_final_safe_wording.csv'),
    Path('reports/revision/tables/table_final_blocker_status.csv'),
    Path('reports/revision/tables/table_final_robustness_claims.csv'),
    Path('reports/revision/manifests/final_evidence_matrix_manifest.json'),
]
missing = [str(path) for path in required if not path.exists() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError('Missing final evidence outputs: ' + '; '.join(missing))
if payload.get('contract_issues'):
    raise RuntimeError('Final evidence has stale contracts: ' + '; '.join(payload['contract_issues']))
if payload.get('missing_inputs'):
    raise RuntimeError('Final evidence has missing inputs: ' + '; '.join(payload['missing_inputs']))
if manifest.get('final_ready_for_rebuttal') is not True or payload.get('final_ready_for_rebuttal') is not True:
    raise RuntimeError('Final evidence is not ready for rebuttal use; inspect final evidence blockers.')
if len(payload.get('evidence_matrix', [])) != 6 or len(payload.get('robustness_claims', [])) != 30:
    raise RuntimeError('Final matrix shape is unexpected.')
c04 = next((row for row in payload['evidence_matrix'] if row.get('claim_id') == 'C04'), {})
if c04.get('evidence_status') == 'complete_probe_available_with_disentanglement_limitation':
    if 'do not support label-aligned morphology-rhythm disentanglement' not in c04.get('safe_wording', ''):
        raise RuntimeError('Representation safe wording is too strong.')
elif 'representation separation remains unproven' not in c04.get('key_numbers', ''):
    raise RuntimeError('Representation state is ambiguous.')
print('Final evidence validation gate: PASS')


## Inventory And Stable Mirror

Refresh the artifact manifest and publish final evidence outputs back to Drive.

In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/final_artifact_inventory.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
    log_path='reports/revision/logs/final_evidence_mirror_publish.log',
)


## Convenience Drive Copies

Copy final reviewer tables to a short Drive path for manual download/opening. These are convenience copies; the canonical artifacts remain under `reports/revision/` and the verified mirror.

In [ ]:
import shutil

FINAL_TABLE_EXPORT_DIR = DRIVE_ROOT / 'final_evidence_tables'
FINAL_TABLE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
final_sources = [
    Path('reports/revision/metrics/final_evidence_matrix.json'),
    Path('reports/revision/tables/table_final_evidence_matrix.csv'),
    Path('reports/revision/tables/table_final_safe_wording.csv'),
    Path('reports/revision/tables/table_final_blocker_status.csv'),
    Path('reports/revision/tables/table_final_robustness_claims.csv'),
    Path('reports/revision/manifests/final_evidence_matrix_manifest.json'),
]
optional_sources = [
    Path('reports/revision/metrics/claim_readiness_gates.json'),
    Path('reports/revision/tables/table_claim_readiness_gates.csv'),
    Path('reports/revision/manifests/claim_readiness_gates_manifest.json'),
    Path('reports/revision/metrics/external_protocol_gate_summary.csv'),
    Path('reports/revision/tables/table_external_ptbxl_metrics.csv'),
    Path('reports/revision/tables/table_external_georgia_metrics.csv'),
    Path('reports/revision/tables/table_external_cpsc2021_metrics.csv'),
    Path('reports/revision/metrics/external_comparator_paired_summary.json'),
    Path('reports/revision/tables/table_external_comparator_paired.csv'),
    Path('reports/revision/metrics/external_comparator_paired_bootstrap_samples.csv'),
    Path('reports/revision/manifests/external_comparator_paired_manifest.json'),
    Path('reports/revision/metrics/group_safe_score_calibration_ptbxl_summary.csv'),
    Path('reports/revision/tables/table_group_safe_score_calibration_ptbxl.csv'),
    Path('reports/revision/metrics/group_safe_score_calibration_ptbxl_bootstrap.json'),
    Path('reports/revision/tables/table_group_safe_score_calibration_ptbxl_coefficients.csv'),
    Path('reports/revision/manifests/group_safe_score_calibration_ptbxl_manifest.json'),
    Path('reports/revision/metrics/true_fewshot_head_ptbxl_summary.csv'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl.csv'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl_paired.csv'),
    Path('reports/revision/metrics/true_fewshot_head_ptbxl_bootstrap.json'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl_coefficients.csv'),
    Path('reports/revision/manifests/true_fewshot_head_ptbxl_manifest.json'),
    Path('reports/revision/metrics/representation_evidence_status.json'),
    Path('reports/revision/metrics/representation_probe_summary.json'),
    Path('reports/revision/tables/table_representation_probe.csv'),
    Path('reports/revision/tables/table_representation_probe_by_fold.csv'),
    Path('reports/revision/tables/table_representation_cka.csv'),
    Path('reports/revision/figures/figure_representation_audit.png'),
    Path('reports/revision/manifests/representation_probe_manifest.json'),
    Path('reports/revision/figures/figure_calibration_audit.png'),
    Path('reports/revision/tables/table_calibration_ci_compact.csv'),
    Path('reports/revision/tables/table_paired_baseline_ci_compact.csv'),
    Path('reports/revision/tables/table_pooling_sensitivity_compact.csv'),
    Path('reports/revision/metrics/pooling_sensitivity_external.csv'),
    Path('reports/revision/tables/table_pooling_sensitivity_across_datasets.csv'),
    Path('reports/revision/tables/table_fold_pca_provenance.csv'),
    Path('reports/revision/tables/table_training_configuration.csv'),
    Path('reports/revision/tables/table_morphology_transform_contract.csv'),
    Path('reports/revision/manifests/morphology_transform_identity_gate.json'),
    Path('reports/revision/manifests/reviewer_completion_input_contract.json'),
    Path('reports/revision/manifests/marked_manuscript_manifest.json'),
]
for src in final_sources + [path for path in optional_sources if path.exists() and path.stat().st_size > 0]:
    if not src.exists():
        raise FileNotFoundError(src)
    shutil.copy2(src, FINAL_TABLE_EXPORT_DIR / src.name)
(FINAL_TABLE_EXPORT_DIR / 'README.md').write_text(
    '# ECG-RAMBA final evidence tables\n\n'
    'These are convenience copies. The canonical provenance remains reports/revision and the verified Drive mirror.\n\n'
    '- Final matrix and safe wording are the source of truth for manuscript/rebuttal wording.\n'
    '- Legacy row-split score calibration is provenance only and is not claim-ready.\n'
    '- Group-safe score calibration is not model-weight adaptation.\n'
    '- True few-shot means new linear classifier heads on frozen encoders, not end-to-end fine-tuning.\n'
    '- PTB-XL/Georgia are separate mapped record-level tasks; CPSC2021 is a separate 10-second AF/AFL mapped-window task.\n'
    '- Representation artifacts are audit/limitation evidence, not proof of disentanglement.\n'
    '- A marked manuscript is complete only when its latexdiff/LaTeX manifest marks it editorial_ready.\n',
    encoding='utf-8',
)
print('Convenience export dir:', FINAL_TABLE_EXPORT_DIR)


## Output Summary

In [ ]:
expected = [
    'reports/revision/metrics/final_evidence_matrix.json',
    'reports/revision/tables/table_final_evidence_matrix.csv',
    'reports/revision/tables/table_final_safe_wording.csv',
    'reports/revision/manifests/final_evidence_matrix_manifest.json',
    'reports/revision/metrics/claim_readiness_gates.json',
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_final_evidence_matrix.csv'),
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_final_safe_wording.csv'),
]
for item in expected:
    path = Path(item)
    print(f'{item}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
print('Claim guidance:')
for key, value in payload.get('claim_guidance', {}).items():
    print(f'- {key}: {value}')
print('Claim readiness:', payload.get('claim_readiness_gates', {}).get('rows', []))
print('Notebook 07 complete. Use table_final_evidence_matrix.csv and table_final_safe_wording.csv as source of truth.')
